In [1]:
# CÉLULA 1
%load_ext autoreload
%autoreload 2

from core.config_parser import SimulationState

yaml_path = "configs/simulation_config.yaml"
state = SimulationState.from_yaml(yaml_path)

print("--- [1] YAML PARSING TEST ---")
print(f"Run Name: {state.identity.run_name} | Solver: {state.identity.solver_type}")
print(f"PBS Queue Selecionada: {state.simulation_control.pbs_queue}")
print(f"Método da Camada Limite: {state.boundary_layer_setup.method}")
print(f"Alvo Y+: {state.mesh_control.target_y_plus}")

[INFO] domain_z is set to 0.0. Processing as a quasi-2D simulation setup.
--- [1] YAML PARSING TEST ---
Run Name: L20_D80_M6_Re3000 | Solver: DNS
PBS Queue Selecionada: mafat14_512_q
Método da Camada Limite: similarity_solution
Alvo Y+: 0.7


In [2]:
# CÉLULA 2
from core.physics_engine import PhysicsEngine

# O PhysicsEngine agora escolhe a estratégia sozinho baseado no estado!
physics_engine = PhysicsEngine(state)
physics_state = physics_engine.calculate_derived_properties()

print("--- [2] PHYSICS ENGINE TEST ---")
print(f"Estratégia Utilizada: {physics_engine.bl_strategy.__class__.__name__}")
print(f"Calculated Pregap No-Slip Length (L2) para atingir Re={state.flow_physics.target_reynolds}: {physics_state.pregap_noslip_length:.4f}")
print(f"Estimated Delta99 no Bordo de Ataque: {physics_state.estimated_delta99:.4f}")
print(f"Calculated Domain Height (H): {physics_state.domain_height:.4f}")

--- [2] PHYSICS ENGINE TEST ---
Estratégia Utilizada: SimilaritySolutionStrategy
Calculated Pregap No-Slip Length (L2) para atingir Re=3000.0: 1514.8226
Estimated Delta99 no Bordo de Ataque: 2.4778
Calculated Domain Height (H): 12.3891


In [3]:
# CÉLULA 3
from core.mesh_planner import MeshPlanner

mesh_planner = MeshPlanner(state, physics_state)
mesh_state = mesh_planner.calculate_mesh_parameters()

print("--- [3] MESH PLANNER TEST ---")
print(f"Tamanho Físico Alvo na Parede (Delta y): {mesh_state.target_y_spacing:.6f}")
print(f"HCP_DELTA (Célula Base do Far-field): {mesh_state.hcp_delta:.4f}")
print(f"Níveis Máximos de Refinamento (LEVEL): {mesh_state.max_refinement_level}")
print(f"Altura de Refinamento Fino: {mesh_state.refinement_height:.4f}")

--- [3] MESH PLANNER TEST ---
Tamanho Físico Alvo na Parede (Delta y): 0.006282
HCP_DELTA (Célula Base do Far-field): 1.2389
Níveis Máximos de Refinamento (LEVEL): 8
Altura de Refinamento Fino: 3.7167


In [4]:
# CÉLULA 4
from core.probe_generator import ProbeGenerator
import os

probe_gen = ProbeGenerator(state, physics_state)
probe_gen.execute()

probe_dir = f"output_simulations/{state.identity.run_name}/probe_coordinates"
print("\n--- [4] PROBE GENERATOR TEST ---")
print(f"Verificando arquivos CSV na pasta: {probe_dir}")
for f in os.listdir(probe_dir):
    # Lendo apenas as primeiras linhas para confirmar a estrutura
    with open(os.path.join(probe_dir, f), 'r') as file:
        lines = file.readlines()
        print(f" - {f} gerado com {len(lines)-1} pontos (coordenadas x,y,z).")

--- Generating Probes ---
[INFO] Generated Space Probe: pregap_noslip -> 1071 points
[INFO] Generated Space Probe: gap_shear_layer -> 9801 points
[INFO] Generated Space Probe: postgap -> 14652 points
[INFO] Generated Time Probe: shear_layer -> 9 points
[INFO] Generated Time Probe: boundary_layer_postgap -> 50 points
--- Probe Generation Complete ---

--- [4] PROBE GENERATOR TEST ---
Verificando arquivos CSV na pasta: output_simulations/L20_D80_M6_Re3000/probe_coordinates
 - SpaceProbe_pregap_noslip.csv gerado com 1071 pontos (coordenadas x,y,z).
 - SpaceProbe_gap_shear_layer.csv gerado com 9801 pontos (coordenadas x,y,z).
 - SpaceProbe_postgap.csv gerado com 14652 pontos (coordenadas x,y,z).
 - TimeProbe_shear_layer.csv gerado com 9 pontos (coordenadas x,y,z).
 - TimeProbe_boundary_layer_postgap.csv gerado com 50 pontos (coordenadas x,y,z).


In [5]:
# CÉLULA 5
from core.template_writer import TemplateWriter

writer = TemplateWriter(state, physics_state, mesh_state, template_dir="templates")
writer.execute()

print("\n--- [5] TEMPLATE WRITER TEST ---")
print(f"Simulação '{state.identity.run_name}' compilada com sucesso!")
print(f"Abra a pasta 'output_simulations/{state.identity.run_name}' para inspecionar os arquivos.")

--- Gerando Arquivos de Simulação ---
[INFO] Gerado arquivo de entrada: surfer.in
[INFO] Gerado arquivo de entrada: stitch.in
[INFO] Gerado arquivo de entrada: transient_charles_ig.in
[INFO] Gerado arquivo de entrada: steady_charles_ig.in
[INFO] Gerado script bash: run-surfer.sh (Fila: mafat14_512_q)
[INFO] Gerado script bash: run-stitch.sh (Fila: mafat14_512_q)
[INFO] Gerado script bash: run-transient-charles.sh (Fila: mafat14_512_q)
[INFO] Gerado script bash: run-steady-charles.sh (Fila: mafat14_512_q)
[WARNING] Template template-move_logs.sh não encontrado em templates
--- Setup de Simulação Finalizado: L20_D80_M6_Re3000 ---

--- [5] TEMPLATE WRITER TEST ---
Simulação 'L20_D80_M6_Re3000' compilada com sucesso!
Abra a pasta 'output_simulations/L20_D80_M6_Re3000' para inspecionar os arquivos.
